# Notebook 05: Fast R-CNN

**Module:** 08 Object Detection  
**Duration:** ~2.5 hrs

---

## Learning Objectives

1. Understand RoI Pooling and shared backbone
2. Single-stage training with multi-task loss
3. Compare speed vs R-CNN

## Fast R-CNN (Girshick, 2015)

**Key innovation:** Run CNN **once** on full image → feature map. Pool features per proposal with **RoI Pooling**.

**RoI Pooling:** Map each proposal onto feature map, divide into fixed grid (e.g. 7×7), max-pool each cell.

**Loss:** Multi-task — softmax cls + Smooth L1 bbox (end-to-end).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import torch
import torch.nn as nn
import torch.nn.functional as F

plt.rcParams['figure.figsize'] = (8, 5)
torch.manual_seed(42)
rng = np.random.default_rng(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
class RoIPool(nn.Module):
    def __init__(self, out_h=7, out_w=7):
        super().__init__()
        self.out_h, self.out_w = out_h, out_w
    def forward(self, x, rois):
        # x: (1,C,H,W), rois: (N,5) [batch_idx,x1,y1,x2,y2] in feature-map coords
        N = rois.shape[0]
        C, H, W = x.shape[1:]
        out = torch.zeros(N, C, self.out_h, self.out_w)
        for i, roi in enumerate(rois):
            _, x1, y1, x2, y2 = roi.tolist()
            x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)
            crop = x[0, :, y1:y2, x1:x2]
            if crop.numel() == 0:
                continue
            crop = crop.unsqueeze(0)
            pooled = F.adaptive_max_pool2d(crop, (self.out_h, self.out_w))
            out[i] = pooled[0]
        return out

feat = torch.randn(1, 64, 32, 32)
rois = torch.tensor([[0, 4, 4, 20, 20], [0, 10, 10, 28, 28]], dtype=torch.float32)
print('RoI pool output:', RoIPool()(feat, rois).shape)

## Speed

~200× faster than R-CNN at test time (shared conv). Still uses external Selective Search for proposals.

---

## Summary

Fast R-CNN shares convolution — one forward pass per image.

**Next:** [06_Faster_R_CNN.ipynb](06_Faster_R_CNN.ipynb)